In [4]:
import xml.etree.ElementTree as ET
import csv
import os

XML_FILE = r"D:\Graduation Project\export\apple_health_export\export.xml"
OUTPUT_FOLDER = r"D:\Graduation Project\export\apple_health_csvs"

# Make sure output folder exists
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Map record types to friendly CSV filenames
TYPE_TO_FILE = {
    "HKQuantityTypeIdentifierBloodGlucose": "blood_glucose.csv",
    "HKQuantityTypeIdentifierInsulinDelivery": "insulin.csv",
    "HKQuantityTypeIdentifierDietaryCarbohydrates": "carbs.csv",
    "HKQuantityTypeIdentifierHeartRate": "heart_rate.csv",
    "HKQuantityTypeIdentifierHeight": "height.csv",
    "HKQuantityTypeIdentifierBodyMass": "weight.csv",
    "HKQuantityTypeIdentifierStepCount": "steps.csv",
    "HKQuantityTypeIdentifierDistanceWalkingRunning": "distance.csv",
    "HKQuantityTypeIdentifierBasalEnergyBurned": "basal_energy.csv",
    "HKQuantityTypeIdentifierPhysicalEffort": "physical_effort.csv"
}

# Open CSV writers for all types
writers = {}
files = {}

for record_type, filename in TYPE_TO_FILE.items():
    file_path = os.path.join(OUTPUT_FOLDER, filename)
    f = open(file_path, "w", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=["timestamp", "value", "unit", "sourceName", "device"])
    writer.writeheader()
    writers[record_type] = writer
    files[record_type] = f

# Stream parse XML
for event, elem in ET.iterparse(XML_FILE, events=("end",)):
    if elem.tag == "Record":
        r_type = elem.attrib.get("type")
        if r_type in TYPE_TO_FILE:
            writer = writers[r_type]
            writer.writerow({
                "timestamp": elem.attrib.get("startDate"),
                "value": elem.attrib.get("value"),
                "unit": elem.attrib.get("unit"),
                "sourceName": elem.attrib.get("sourceName", ""),
                "device": elem.attrib.get("device", "")
            })
        elem.clear()

# Close all files
for f in files.values():
    f.close()

print(f"All CSVs saved in: {OUTPUT_FOLDER}")


All CSVs saved in: D:\Graduation Project\export\apple_health_csvs


In [5]:
with open(r"D:\Graduation Project\export\apple_health_export\export.xml", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if "InsulinDelivery" in line:
            print(f"Found Insulin record at line {i}:")
            print(line.strip())
            break
    else:
        print("No insulin records found in the XML.")


No insulin records found in the XML.


In [6]:
import xml.etree.ElementTree as ET

XML_FILE = r"D:\Graduation Project\export\apple_health_export\export.xml"

types = set()

for event, elem in ET.iterparse(XML_FILE, events=("end",)):
    if elem.tag == "Record":
        t = elem.attrib.get("type")
        if t:
            types.add(t)
    elem.clear()

# Print all types
for t in sorted(types):
    print(t)


HKCategoryTypeIdentifierAppleStandHour
HKCategoryTypeIdentifierAudioExposureEvent
HKCategoryTypeIdentifierFatigue
HKCategoryTypeIdentifierHeadphoneAudioExposureEvent
HKCategoryTypeIdentifierHighHeartRateEvent
HKCategoryTypeIdentifierLowCardioFitnessEvent
HKCategoryTypeIdentifierMindfulSession
HKCategoryTypeIdentifierShortnessOfBreath
HKCategoryTypeIdentifierSleepAnalysis
HKDataTypeSleepDurationGoal
HKQuantityTypeIdentifierActiveEnergyBurned
HKQuantityTypeIdentifierAppleExerciseTime
HKQuantityTypeIdentifierAppleSleepingWristTemperature
HKQuantityTypeIdentifierAppleStandTime
HKQuantityTypeIdentifierAppleWalkingSteadiness
HKQuantityTypeIdentifierBasalEnergyBurned
HKQuantityTypeIdentifierBloodGlucose
HKQuantityTypeIdentifierBodyMass
HKQuantityTypeIdentifierDistanceWalkingRunning
HKQuantityTypeIdentifierEnvironmentalAudioExposure
HKQuantityTypeIdentifierEnvironmentalSoundReduction
HKQuantityTypeIdentifierFlightsClimbed
HKQuantityTypeIdentifierHeadphoneAudioExposure
HKQuantityTypeIdentifierH

In [ ]:
import pandas as pd
import os
from functools import reduce

folder = r"D:\Graduation Project\export\apple_health_csvs"
output_csv = r"D:\Graduation Project\export\apple_health_merged.csv"

files = [
    "basal_energy.csv",
    "blood_glucose.csv",
    "carbs.csv",          
    "distance.csv",
    "heart_rate.csv",
    "height.csv",
    "insulin.csv",
    "physical_effort.csv",
    "steps.csv",
    "weight.csv"
]

dfs = []

for file in files:
    path = os.path.join(folder, file)
    if os.path.exists(path):
        df = pd.read_csv(path)
        # Parse timestamps and **remove timezone info 3lshan kan bygeeb error fa sabetnah**
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert(None)
        # Rename value column to match the signal name
        signal_name = file.replace(".csv", "")
        df = df[['timestamp', 'value']].rename(columns={'value': signal_name})
        dfs.append(df)
    else:
        print(f"Warning: {file} not found in folder!")

# Merge all CSVs on timestamp (outer join)
merged_df = reduce(lambda left, right: pd.merge(left, right, on='timestamp', how='outer'), dfs)

# Sort by timestamp
merged_df = merged_df.sort_values('timestamp')

# Save final merged dataset
merged_df.to_csv(output_csv, index=False)
print(f"Merged dataset saved to: {output_csv}")


Merged dataset saved to: D:\Graduation Project\export\apple_health_merged.csv
